# Wariant D – analiza dynamiki z modelem JEV

Zamiana generatywnego LLM z wariantu D na model decyzyjny **JEV** (`jev-1.13-free`, API opencode.ai).
Dla każdej pary *uczestnik × slajd* JEV ocenia trzy wymiary w skali 0–2:
`engagement_level`, `confidence_level`, `communication_style`.

**Wejście:** `parsed_transcripts.jsonl` (z wariantu D).
**Wyjście:** `jev_results.jsonl` (wyniki), `jev_errors.jsonl` (błędy zapytań).

**Przed uruchomieniem:** w tym folderze plik `.env` z linią `OPENCODE_API_KEY=...`
(plik jest w `.gitignore`). Wymagane pakiety: `requests`, `python-dotenv`.

Notebook można uruchamiać wielokrotnie: wyniki zapisują się po każdym rekordzie,
a kolejne uruchomienie pomija rekordy już ocenione.

## 1. Konfiguracja

In [1]:
import json
import os
import time
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(Path(".env"))
API_KEY = os.getenv("OPENCODE_API_KEY")
if not API_KEY:
    raise ValueError("Brak OPENCODE_API_KEY – dodaj go do pliku .env w tym folderze.")

JEV_URL = "https://opencode.ai/inference/systemone/v1/systemone"
JEV_MODEL = "jev-1.13-free"

TRANSCRIPTS_PATH = Path("parsed_transcripts.jsonl")
RESULTS_PATH = Path("jev_results.jsonl")
ERRORS_PATH = Path("jev_errors.jsonl")

## 2. Dane – tylko wypowiedzi uczestników

`parsed_transcripts.jsonl` zawiera też wypowiedzi badaczy (`PK`, `MW`, `MK`, `MZ`) – odrzucamy je.
Plik `PK_DE_11-12` zawiera dwóch uczestników.

Uwaga: w pliku brakuje 7 transkryptów prowadzonych przez badacza DR (`DR_IT_05`, `DR_SSH_01–04`, `DR_SSH_06–07`),
więc analizą objętych jest 32 z 39 uczestników.

In [2]:
def is_participant(r):
    stem = r["source_file"].rsplit(".", 1)[0]
    pid = r["participant_id"]
    if stem == "PK_DE_11-12":
        return pid in ("PK_DE_11", "PK_DE_12")
    return pid == stem


def record_key(r):
    return (r["source_file"], r["participant_id"], r["slide_id"])


def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


all_records = read_jsonl(TRANSCRIPTS_PATH)
transcripts = [r for r in all_records if is_participant(r)]

print("Wszystkie rekordy:", len(all_records))                                     # 1781
print("Rekordy uczestników:", len(transcripts))                                   # 932
print("Liczba uczestników:", len({r["participant_id"] for r in transcripts}))     # 32

Wszystkie rekordy: 1781
Rekordy uczestników: 932
Liczba uczestników: 32


## 3. Zapytanie do JEV

Pytania są takie same jak przy wygenerowaniu dotychczasowych wyników – nie zmieniać ich w trakcie,
bo wyniki przestaną być porównywalne.

In [3]:
QUESTIONS = {
    "engagement_level": {
        "type": "score",
        "instructions": (
            "Assess the participant's level of engagement "
            "based only on the transcript. Consider active "
            "involvement, attention and willingness to contribute."
        ),
        "criteria": ["Low engagement", "Moderate engagement", "High engagement"],
    },
    "confidence_level": {
        "type": "score",
        "instructions": (
            "Assess the participant's expressed confidence "
            "in their statements and decisions. Do not assess "
            "the model's confidence."
        ),
        "criteria": ["Low expressed confidence", "Moderate expressed confidence", "High expressed confidence"],
    },
    "communication_style": {
        "type": "score",
        "instructions": (
            "Assess the participant's communication style "
            "based on structure and elaboration of statements."
        ),
        "criteria": [
            "Concise and direct communication",
            "Balanced communication",
            "Elaborative and reflective communication",
        ],
    },
}


def analyze_with_jev(record):
    """Ocena jednego rekordu (uczestnik × slajd) przez JEV."""
    response = requests.post(
        JEV_URL,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json={"model": JEV_MODEL, "state": record["aggregated_text"], "questions": QUESTIONS},
        timeout=60,
    )
    response.raise_for_status()
    result = response.json()

    scores = {
        dim: {
            "score": ans["score"],
            "normalized_score": ans["score"] / 2,
            "confidence": ans["confidence"],
            "probabilities": ans["probabilities"],
        }
        for dim, ans in result["answers"].items()
    }
    return {
        "participant_id": record["participant_id"],
        "slide_id": record["slide_id"],
        "source_file": record["source_file"],
        "utterance_count": record["utterance_count"],
        "token_count": record["token_count"],
        "scores": scores,
        "usage": result.get("usage", {}),
    }

## 4. Porządkowanie zapisanych wyników

Usuwa z `jev_results.jsonl` duplikaty (zostaje pierwsza ocena) i ewentualne oceny badaczy.
Bezpieczne do uruchamiania wielokrotnie.

In [4]:
saved = read_jsonl(RESULTS_PATH)
clean, seen = [], set()
for r in saved:
    if is_participant(r) and record_key(r) not in seen:
        clean.append(r)
        seen.add(record_key(r))

if len(clean) != len(saved):
    RESULTS_PATH.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in clean), encoding="utf-8")

print(f"Wyniki: {len(saved)} -> {len(clean)} (usunięto {len(saved) - len(clean)})")

Wyniki: 0 -> 0 (usunięto 0)


## 5. Uruchomienie JEV (kolejny batch)

Każde uruchomienie ocenia `BATCH_SIZE` kolejnych, jeszcze nieocenionych rekordów.
`BATCH_SIZE = None` – wszystkie pozostałe naraz (ok. 900 zapytań, kilkadziesiąt minut).

In [8]:
BATCH_SIZE = 932
DELAY_SECONDS = 1

processed = {record_key(r) for r in read_jsonl(RESULTS_PATH)}
pending = [r for r in transcripts if record_key(r) not in processed]
batch = pending if BATCH_SIZE is None else pending[:BATCH_SIZE]

print(f"Do oceny łącznie: {len(pending)}, w tym batchu: {len(batch)}")

ok = errors = 0
for i, record in enumerate(batch, start=1):
    print(f"[{i}/{len(batch)}] {record['participant_id']} | {record['slide_id']}", end=" ")
    try:
        output = analyze_with_jev(record)
        with open(RESULTS_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(output, ensure_ascii=False) + "\n")
        ok += 1
        print("-> OK")
    except Exception as e:
        with open(ERRORS_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({**dict(zip(("source_file", "participant_id", "slide_id"), record_key(record))),
                                "error": str(e)}, ensure_ascii=False) + "\n")
        errors += 1
        print("-> ERROR:", e)
    if i < len(batch):
        time.sleep(DELAY_SECONDS)

print(f"\nSukcesy: {ok}, błędy: {errors}, pozostało: {len(pending) - ok}")

Do oceny łącznie: 657, w tym batchu: 657
[1/657] MZ_SSH_07 | __S3__ -> OK
[2/657] MZ_SSH_07 | __S4__ -> OK
[3/657] MZ_SSH_07 | __S5__ -> OK
[4/657] MZ_SSH_07 | __S6__ -> OK
[5/657] MZ_SSH_07 | __S7__ -> OK
[6/657] MZ_SSH_07 | __S8__ -> OK
[7/657] MZ_SSH_07 | __S9__ -> OK
[8/657] MZ_SSH_07 | __S10__ -> OK
[9/657] MZ_SSH_07 | __S11__ -> OK
[10/657] MZ_SSH_07 | __S12__ -> OK
[11/657] MZ_SSH_07 | __S13__ -> OK
[12/657] MZ_SSH_07 | __S14__ -> OK
[13/657] MZ_SSH_07 | __S88__ -> OK
[14/657] MZ_SSH_07 | __S99__ -> OK
[15/657] PK_DE_01 | __S1__ -> OK
[16/657] PK_DE_01 | __S3__ -> OK
[17/657] PK_DE_01 | __S4__ -> OK
[18/657] PK_DE_01 | __S5__ -> OK
[19/657] PK_DE_01 | __S6__ -> OK
[20/657] PK_DE_01 | __S7__ -> OK
[21/657] PK_DE_01 | __S8__ -> OK
[22/657] PK_DE_01 | __S9__ -> OK
[23/657] PK_DE_01 | __S10__ -> OK
[24/657] PK_DE_01 | __S11__ -> OK
[25/657] PK_DE_01 | __S12__ -> OK
[26/657] PK_DE_01 | __S13__ -> OK
[27/657] PK_DE_01 | __S14__ -> OK
[28/657] PK_DE_01 | __S99__ -> OK
[29/657] PK_DE_02

## 6. Kontrola jakości wyników

In [6]:
EXPECTED_DIMENSIONS = set(QUESTIONS)

results = read_jsonl(RESULTS_PATH)
issues, seen = [], set()

for i, item in enumerate(results):
    key = record_key(item)
    if key in seen:
        issues.append(f"Duplikat [{i}]: {key}")
    seen.add(key)
    if not is_participant(item):
        issues.append(f"Ocena badacza [{i}]: {key}")

    scores = item.get("scores", {})
    if set(scores) != EXPECTED_DIMENSIONS:
        issues.append(f"Brak wymiarów [{i}]")
    for dim, d in scores.items():
        if d.get("score") is None or not 0 <= d["score"] <= 2:
            issues.append(f"Zły score [{i}] {dim}")
        if d.get("confidence") is None or not 0 <= d["confidence"] <= 1:
            issues.append(f"Zła confidence [{i}] {dim}")
        if d.get("probabilities") and abs(sum(d["probabilities"].values()) - 1) > 0.02:
            issues.append(f"Prawdopodobieństwa nie sumują się do 1 [{i}] {dim}")

print("Wyniki:", len(results), "| unikalne:", len(seen),
      "| uczestnicy:", len({r["participant_id"] for r in results}),
      f"| postęp: {len(seen)}/{len(transcripts)}")
print("Problemy:", len(issues))
for issue in issues[:20]:
    print("-", issue)
if not issues:
    print("QC PASSED")

Wyniki: 20 | unikalne: 20 | uczestnicy: 2 | postęp: 20/932
Problemy: 0
QC PASSED


## 7. Wyjaśnienia (TODO)

Do zrobienia: wykorzystać kod z wariantu D (`XAIExplanation`, `Plot`) na wynikach JEV zamiast odpowiedzi LLM.

In [12]:
import json
from collections import defaultdict
from pathlib import Path

def slide_order(s):
    try:
        return int(s.strip("_S"))
    except ValueError:
        return 10**6

# Oceny JEV -> format, którego oczekuje XAIExplanation (jak llm_responses.json)
by_participant = defaultdict(dict)
for r in read_jsonl(RESULTS_PATH):
    profile = {dim: round(v["normalized_score"], 2) for dim, v in r["scores"].items()}
    details = {dim: {"pewnosc_modelu": v["confidence"], "rozklad": v["probabilities"]}
               for dim, v in r["scores"].items()}
    by_participant[r["participant_id"]][r["slide_id"]] = [json.dumps(
        {"participant_id": r["participant_id"], "profile": profile, "jev": details},
        ensure_ascii=False)]

jev_for_xai = {p: dict(sorted(s.items(), key=lambda kv: slide_order(kv[0])))
               for p, s in by_participant.items()}
Path("jev_responses_for_xai.json").write_text(
    json.dumps(jev_for_xai, ensure_ascii=False, indent=2), encoding="utf-8")

# Transkrypty tylko uczestników (XAIExplanation brałby też wypowiedzi badaczy)
Path("parsed_transcripts_participants.jsonl").write_text(
    "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in transcripts), encoding="utf-8")

print("Uczestnicy z ocenami JEV:", len(jev_for_xai))

Uczestnicy z ocenami JEV: 32


In [13]:
from src.XAIExplanation import XAIExplanation
from src.LocalLLM import LocalLLM

out_dir = Path("xai_explanations_jev")
out_dir.mkdir(exist_ok=True)
llm = LocalLLM()

for pid in jev_for_xai:
    save_path = out_dir / f"{pid}_explanation.json"
    if save_path.exists():
        continue
    xai = XAIExplanation(llm=llm)   # nowy obiekt dla każdego uczestnika
    explanations = xai.explain(
        llm_response_path="jev_responses_for_xai.json",
        transcript_path="parsed_transcripts_participants.jsonl",
        participant_id=pid,
    )
    if explanations:
        save_path.write_text(json.dumps(explanations, ensure_ascii=False, indent=2), encoding="utf-8")
        print(pid, "->", len(explanations), "wyjaśnień")

Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character maps to <undefined>
Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character maps to <undefined>
Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character maps to <undefined>
Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character maps to <undefined>
Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character maps to <undefined>
Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character maps to <undefined>
Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character maps to <undefined>
Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character maps to <undefined>
Error reading transcript: 'charmap' codec can't decode byte 0x81 in position 1063: character map